In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# LaB₆ — neutron powder, constant wavelength, FCJ asymmetry

In [2]:
import easydiffraction as edi
from easydiffraction import ExperimentFactory
from easydiffraction import StructureFactory
from easydiffraction.analysis import verification as verify

## Build the project

In [3]:
project = edi.Project()

## Define the structure

In [4]:
structure = StructureFactory.from_scratch(name='lab6')
structure.space_group.name_h_m = 'P m -3 m'  # FullProf Space group symbol
structure.cell.length_a = 4.156885  # FullProf a
structure.atom_sites.create(
    id='La',  # FullProf Atom
    type_symbol='La',  # FullProf Typ
    fract_x=0.0,  # FullProf X
    fract_y=0.0,  # FullProf Y
    fract_z=0.0,  # FullProf Z
    adp_type='Biso',  # FullProf Biso
    adp_iso=0.32249,  # FullProf Biso
)
structure.atom_sites.create(
    id='B',  # FullProf Atom
    type_symbol='11B',  # FullProf "B11"
    fract_x=0.19978,  # FullProf X
    fract_y=0.5,  # FullProf Y
    fract_z=0.5,  # FullProf Z
    adp_type='Biso',  # FullProf Biso
    adp_iso=0.16910,  # FullProf Biso
)

project.structures.add(structure)

## Load the FullProf reference

In [5]:
FULLPROF_PROJECT_DIR = 'pd-neut-cwl_tch-fcj_lab6'
FULLPROF_PRF_FILE = 'ECH0030684_LaB6_1p622A_noAbs.prf'
FULLPROF_BAC_FILE = 'ECH0030684_LaB6_1p622A_noAbs.bac'
FULLPROF_ZERO = -0.21148  # FullProf Zero
FULLPROF_SCALE = 44.51785  # FullProf Scale
FULLPROF_WAVELENGTH = 1.622528  # FullProf Lambda
FULLPROF_U = 0.089670  # FullProf U
FULLPROF_V = -0.375862  # FullProf V
FULLPROF_W = 0.476309  # FullProf W
FULLPROF_X = 0.0  # FullProf X
FULLPROF_Y = 0.052528  # FullProf Y
FULLPROF_SYCOS = 0.05290  # FullProf SyCos
FULLPROF_SYSIN = 0.09073  # FullProf SySin
FULLPROF_S_L = 0.08000  # FullProf S_L
FULLPROF_D_L = 0.08000  # FullProf D_L

x, calc_fullprof = verify.load_fullprof_calc_profile(
    FULLPROF_PROJECT_DIR,
    FULLPROF_PRF_FILE,
    FULLPROF_BAC_FILE,
    FULLPROF_ZERO,
)

## Create the experiment

In [6]:
experiment = ExperimentFactory.from_scratch(
    name='lab6',
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
    scattering_type='bragg',
)
verify.set_reference_as_measured(experiment, x, calc_fullprof)

experiment.linked_structures.create(structure_id='lab6', scale=FULLPROF_SCALE)

experiment.instrument.setup_wavelength = FULLPROF_WAVELENGTH
experiment.instrument.calib_twotheta_offset = FULLPROF_ZERO

experiment.peak.broad_gauss_u = FULLPROF_U
experiment.peak.broad_gauss_v = FULLPROF_V
experiment.peak.broad_gauss_w = FULLPROF_W
experiment.peak.broad_lorentz_x = FULLPROF_X
experiment.peak.broad_lorentz_y = FULLPROF_Y
# Engine-specific corrections are applied in each engine's section below:
# SyCos/SySin (cryspy only) and the FCJ S_L/D_L asymmetry (crysfml only).

project.experiments.add(experiment)

## edi-cryspy VS FullProf

In [7]:
experiment.calculator.type = 'cryspy'

experiment.instrument.calib_sample_displacement = FULLPROF_SYCOS
experiment.instrument.calib_sample_transparency = FULLPROF_SYSIN

project.analysis.calculate()
calc_ed_cryspy = experiment.data.intensity_calc

project.display.pattern_comparison(
    'lab6',
    reference=calc_fullprof,
    candidate=calc_ed_cryspy,
    reference_label='FullProf',
    candidate_label='edi-cryspy',
)

Calculator for experiment 'lab6' already set to


cryspy


## Fit edi-cryspy to FullProf

In [8]:
experiment.linked_structures['lab6'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True
experiment.instrument.calib_sample_displacement.free = True
experiment.instrument.calib_sample_transparency.free = True

project.analysis.fit()
project.display.fit.results()

project.analysis.calculate()
calc_ed_cryspy_refined = experiment.data.intensity_calc

project.display.pattern_comparison(
    'lab6',
    reference=calc_fullprof,
    candidate=calc_ed_cryspy_refined,
    reference_label='FullProf',
    candidate_label='edi-cryspy (refined)',
)

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'lab6' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),2.46
4,🔁 Iterations,46
5,📏 Goodness-of-fit (reduced χ²),74787.93
6,"📏 R-factor (Rf, %)",10.82
7,"📏 R-factor squared (Rf², %)",8.98
8,"📏 Weighted R-factor (wR, %)",8.98


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lab6,linked_structure,lab6,scale,,44.5179,42.8167,0.0684,3.82 % ↓
2,lab6,instrument,,twotheta_offset,deg,-0.2115,-0.0768,0.0038,63.66 % ↓
3,lab6,instrument,,sample_displacement,deg,0.0529,-0.2747,0.0029,619.28 % ↓
4,lab6,instrument,,sample_transparency,deg,0.0907,0.1518,0.0040,67.36 % ↑


## edi-crysfml VS FullProf

In [9]:
experiment.calculator.type = 'crysfml'

experiment.linked_structures['lab6'].scale = FULLPROF_SCALE

experiment.peak.type = 'thompson-cox-hastings'
experiment.instrument.calib_twotheta_offset = FULLPROF_ZERO
experiment.peak.broad_gauss_u = FULLPROF_U
experiment.peak.broad_gauss_v = FULLPROF_V
experiment.peak.broad_gauss_w = FULLPROF_W
experiment.peak.broad_lorentz_x = FULLPROF_X
experiment.peak.broad_lorentz_y = FULLPROF_Y
experiment.peak.asym_fcj_1 = FULLPROF_S_L
experiment.peak.asym_fcj_2 = FULLPROF_D_L

project.analysis.calculate()
calc_ed_crysfml = experiment.data.intensity_calc

project.display.pattern_comparison(
    'lab6',
    reference=calc_fullprof,
    candidate=calc_ed_crysfml,
    reference_label='FullProf',
    candidate_label='edi-crysfml',
)

Calculator for experiment 'lab6' changed to


crysfml


⚠️ Switching peak profile type adds these settings with defaults:                                                                 
   • asym_fcj_1=0.0                                                                                                               
   • asym_fcj_2=0.0                                                                                                               


⚠️ Switching peak profile type resets these settings to defaults:                                                                 
   • broad_gauss_u: 0.08967 -> 0.01                                                                                               
   • broad_gauss_v: -0.375862 -> -0.01                                                                                            
   • broad_gauss_w: 0.476309 -> 0.02                                                                                              
   • broad_lorentz_y: 0.052528 -> 0.0                                                                                             


Peak profile type for experiment 'lab6' changed to


thompson-cox-hastings


## Fit edi-crysfml to FullProf

In [10]:
experiment.linked_structures['lab6'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True

experiment.instrument.calib_sample_displacement.free = False
experiment.instrument.calib_sample_transparency.free = False

project.analysis.fit()
project.display.fit.results()

project.analysis.calculate()
calc_ed_crysfml_refined = experiment.data.intensity_calc

project.display.pattern_comparison(
    'lab6',
    reference=calc_fullprof,
    candidate=calc_ed_crysfml_refined,
    reference_label='FullProf',
    candidate_label='edi-crysfml (refined)',
)

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'lab6' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),5.22
4,🔁 Iterations,24
5,📏 Goodness-of-fit (reduced χ²),669180.65
6,"📏 R-factor (Rf, %)",28.69
7,"📏 R-factor squared (Rf², %)",26.88
8,"📏 Weighted R-factor (wR, %)",26.88


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lab6,linked_structure,lab6,scale,,44.5179,60.4889,0.2993,35.88 % ↑
2,lab6,instrument,,twotheta_offset,deg,-0.2115,-0.2049,0.0013,3.12 % ↓


## Agreement check

In [11]:
verify.assert_patterns_agree(
    [
        ('cryspy vs FullProf', calc_fullprof, calc_ed_cryspy),
        ('crysfml vs FullProf', calc_fullprof, calc_ed_crysfml),
    ],
    raise_on_failure=False,
)

,Comparison,Metric,Expected,Actual,OK
1,cryspy vs FullProf,Profile diff (%),< 2.5,22.93,❌
2,,Max deviation (%),< 6,18.27,❌
3,,Area ratio,0.99 to 1.01,1.0029,✅
4,,Shape correlation,> 0.999,0.9733,❌
5,crysfml vs FullProf,Profile diff (%),< 2.5,38.02,❌
6,,Max deviation (%),< 6,44.16,❌
7,,Area ratio,0.99 to 1.01,0.6921,❌
8,,Shape correlation,> 0.999,0.9553,❌


False